### Sample code to load the sporc dataset.

The problem is that the episode line entries in the episodeLevelData.jsonl.gz have different types, and for the datasets library to infer the correct common types, it has to load a lot a lot of entries even with the streaming option on.

What follows is the simples way I found to load the dataset, which allows for preprocesing in local RAM. 

In [ ]:
!pip install matplotlib huggingface_hub datasets pandas

In [ ]:
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download, login
from datasets import load_dataset
import gzip
import json
from pprint import pprint
import itertools
import pandas as pd

In [ ]:
# You have to create a huggingface account and accept the dataset terms of use
# After that, get an access token with "read" permissions and paste it below
#login("PUT YOUR TOKEN HERE")

In [2]:
episode_path = hf_hub_download(
    repo_id="blitt/SPoRC",
    filename="episodeLevelData.jsonl.gz",
    repo_type="dataset",
    local_dir="data",
)
speaker_path = hf_hub_download(
    repo_id="blitt/SPoRC",
    filename="speakerTurnData.jsonl.gz",
    repo_type="dataset",
    local_dir="data",
)

In [3]:
def load_episodes_to_df(limit=100_000):
    episodes = []
    with gzip.open(episode_path, 'rt') as f:
        for line in itertools.islice(f, limit):
            episodes.append(json.loads(line))
    return pd.DataFrame(episodes)

episode_df = load_episodes_to_df(limit=10_000)
print("Loaded episode_df:", episode_df.shape)

Loaded episode_df: (10000, 42)


In [ ]:
speaker_ds = load_dataset("json", data_files=speaker_path, split="train", streaming=True)

def load_speakers_to_df(limit=600_000):
    rows = []
    for i, sample in enumerate(speaker_ds):
        if i >= limit:
            break
        rows.append(sample)
    return pd.DataFrame(rows)

speaker_df = load_speakers_to_df(limit=200_000)
print("Loaded speaker_df:", speaker_df.shape)

Loaded speaker_df: (200000, 15)


The dataset should be loaded in both speaker_df and episode_df at this point, the following is some simple processing, you may change with your own code.

In [5]:
print(speaker_df.columns)
print(episode_df.columns)

Index(['mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 'mfcc4_sma3Mean',
       'F0semitoneFrom27.5Hz_sma3nzMean', 'F1frequency_sma3nzMean', 'turnText',
       'speaker', 'startTime', 'endTime', 'duration', 'mp3url', 'turnCount',
       'inferredSpeakerRole', 'inferredSpeakerName'],
      dtype='object')
Index(['transcript', 'rssUrl', 'epTitle', 'epDescription', 'mp3url',
       'podTitle', 'lastUpdate', 'itunesAuthor', 'itunesOwnerName', 'explicit',
       'imageUrl', 'language', 'createdOn', 'host', 'podDescription',
       'category1', 'category2', 'category3', 'category4', 'category5',
       'category6', 'category7', 'category8', 'category9', 'category10',
       'oldestEpisodeDate', 'episodeDateLocalized', 'durationSeconds',
       'hostPredictedNames', 'numUniqueHosts', 'guestPredictedNames',
       'numUniqueGuests', 'neitherPredictedNames', 'numUniqueNeithers',
       'mainEpSpeakers', 'numMainSpeakers', 'hostSpeakerLabels',
       'guestSpeakerLabels', 'overlapPropTurnC

In [6]:
# Do the merge only for some columns to avoid memory issues
small_speaker_df = speaker_df[["mp3url", "inferredSpeakerName", "turnText", "startTime", "endTime"]]
small_episode_df = episode_df[["mp3url", "podTitle", "epTitle", "category1", "durationSeconds"]]
combined_df = pd.merge(small_speaker_df, small_episode_df, on="mp3url", how="left")
print("Combined DataFrame:", combined_df.shape)

Combined DataFrame: (200000, 9)


In [7]:
combined_df

,mp3url,inferredSpeakerName,turnText,startTime,endTime,podTitle,epTitle,category1,durationSeconds
0,https://www.buzzsprout.com/783020/4252475-best...,Simon Shapiro,I'm Simon Shapiro and this is Sing Out Speak ...,0.00,60.00,SingOut SpeakOut,Best of SingOut SpeakOut No.3,music,803.0
1,https://www.buzzsprout.com/783020/4165286-it-s...,Simon Shapiro,I'm Simon Shapiro and this is Sing Out Speak ...,0.00,78.16,SingOut SpeakOut,It's All Gone,music,360.0
2,https://www.buzzsprout.com/783020/4165286-it-s...,NO_INFERRED_SPEAKER,Music] [Music] [Music] [Music] [Music] [,78.16,115.31,SingOut SpeakOut,It's All Gone,music,360.0
3,https://www.buzzsprout.com/783020/4165286-it-s...,NO_INFERRED_SPEAKER,Music] [Music] [Music] [Music] [Music] [Music]...,115.31,360.16,SingOut SpeakOut,It's All Gone,music,360.0
4,https://www.buzzsprout.com/783020/3983942-toda...,Simon Shapiro,I'm Simon Shapiro and this is Sing Out Speak ...,0.00,36.99,SingOut SpeakOut,Today Is Yesterday,music,416.0
...,...,...,...,...,...,...,...,...,...
199995,https://anchor.fm/s/a9cd2a4/podcast/play/14615...,NO_INFERRED_SPEAKER,.,1689.51,1689.76,Strength for Today's Pastor,072 Acts 2 Distinctives Series- the Importance...,religion,2818.0
199996,https://anchor.fm/s/a9cd2a4/podcast/play/14615...,NO_INFERRED_SPEAKER,"Mm hmm, right. Well what about preparation fo...",1689.76,1716.30,Strength for Today's Pastor,072 Acts 2 Distinctives Series- the Importance...,religion,2818.0
199997,https://anchor.fm/s/a9cd2a4/podcast/play/14615...,NO_INFERRED_SPEAKER,head,1716.30,1716.67,Strength for Today's Pastor,072 Acts 2 Distinctives Series- the Importance...,religion,2818.0
199998,https://anchor.fm/s/a9cd2a4/podcast/play/14615...,Ed Compion,"? Yeah, I don't want to say we have this maste...",1716.67,1785.26,Strength for Today's Pastor,072 Acts 2 Distinctives Series- the Importance...,religion,2818.0


In [10]:
interesting_df = combined_df[combined_df["category1"] == "society"][combined_df["durationSeconds"] > 10 * 60]
interesting_df

C:\Users\vlad\AppData\Local\Temp\ipykernel_7796\1823820711.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  interesting_df = combined_df[combined_df["category1"] == "society"][combined_df["durationSeconds"] > 10 * 60]


,mp3url,inferredSpeakerName,turnText,startTime,endTime,podTitle,epTitle,category1,durationSeconds
880,https://anchor.fm/s/11abe148/podcast/play/1565...,NO_INFERRED_SPEAKER,(upbeat music) Welcome to this episode of Thi...,0.00,50.70,This Epoch Life,Shifting Cultures,society,1315.0
881,https://anchor.fm/s/11abe148/podcast/play/1565...,NO_INFERRED_SPEAKER,Absolutely.,50.70,52.04,This Epoch Life,Shifting Cultures,society,1315.0
882,https://anchor.fm/s/11abe148/podcast/play/1565...,NO_INFERRED_SPEAKER,So it came about,52.04,55.52,This Epoch Life,Shifting Cultures,society,1315.0
883,https://anchor.fm/s/11abe148/podcast/play/1565...,NO_INFERRED_SPEAKER,right in the middle of the,55.52,56.48,This Epoch Life,Shifting Cultures,society,1315.0
884,https://anchor.fm/s/11abe148/podcast/play/1565...,NO_INFERRED_SPEAKER,"Me Too movement at that time. I, my backgroun...",56.48,88.91,This Epoch Life,Shifting Cultures,society,1315.0
...,...,...,...,...,...,...,...,...,...
194736,https://anchor.fm/s/4e63b98/podcast/play/14446...,Stella Damassis,"In this episode, I will talk about menstrual ...",16.40,19.76,Excuse My African,EP 74 - Menstrual Hygiene in Africa,society,994.0
194737,https://anchor.fm/s/4e63b98/podcast/play/14446...,NO_INFERRED_SPEAKER,". Now, menstrual cycle for women is one topic",19.76,27.20,Excuse My African,EP 74 - Menstrual Hygiene in Africa,society,994.0
194738,https://anchor.fm/s/4e63b98/podcast/play/14446...,Stella Damassis,that a lot of people don't pay attention to. ...,27.20,691.99,Excuse My African,EP 74 - Menstrual Hygiene in Africa,society,994.0
194739,https://anchor.fm/s/4e63b98/podcast/play/14446...,NO_INFERRED_SPEAKER,you have,691.99,692.53,Excuse My African,EP 74 - Menstrual Hygiene in Africa,society,994.0


In [11]:
# Combine consecutive turns if they are by same speaker and <0.03s apart
# First group by mp3url to handle episodes separately
def merge_turns(group_df):
    # Sort by start time
    group_df = group_df.sort_values('startTime')
    
    # Initialize group ID
    current_group = 0
    group_ids = []
    current_speaker = None
    prev_end = None
    
    for idx, row in group_df.iterrows():
        # Start new group if different speaker or gap too large
        if (current_speaker != row['inferredSpeakerName'] or 
            (prev_end is not None and row['startTime'] - prev_end > 0.02)):
            current_group += 1
        
        group_ids.append(current_group)
        current_speaker = row['inferredSpeakerName']
        prev_end = row['endTime']
    
    group_df['merge_group'] = group_ids
    return group_df

# Apply to each podcast episode
interesting_df = interesting_df.groupby('mp3url').apply(merge_turns).reset_index(drop=True)

# Then merge the text for each group
merged_groups = interesting_df.groupby(['mp3url', 'merge_group']).agg({
    'turnText': ' '.join,
    'inferredSpeakerName': 'first',
    'startTime': 'min',
    'endTime': 'max',
    'podTitle': 'first',
    'epTitle': 'first',
    'category1': 'first',
    'durationSeconds': 'first'
}).reset_index()

merged_df = merged_groups
merged_df

C:\Users\vlad\AppData\Local\Temp\ipykernel_7796\4124566386.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  interesting_df = interesting_df.groupby('mp3url').apply(merge_turns).reset_index(drop=True)


,mp3url,merge_group,turnText,inferredSpeakerName,startTime,endTime,podTitle,epTitle,category1,durationSeconds
0,http://podtrac.com/pts/redirect.mp3/traffic.li...,1,Join us today during the Jeep Celebration eve...,NO_INFERRED_SPEAKER,0.00,37.62,Pretty Much Pop: A Culture Podcast,PMP#42: Star Trek Lives Long and Prospers (Int...,society,3390.0
1,http://podtrac.com/pts/redirect.mp3/traffic.li...,2,] This is pretty much Pop a Culture Podcast,Mark Lindstamire,37.62,41.33,Pretty Much Pop: A Culture Podcast,PMP#42: Star Trek Lives Long and Prospers (Int...,society,3390.0
2,http://podtrac.com/pts/redirect.mp3/traffic.li...,3,making nutritive paste out of the sludge of e...,NO_INFERRED_SPEAKER,41.33,48.28,Pretty Much Pop: A Culture Podcast,PMP#42: Star Trek Lives Long and Prospers (Int...,society,3390.0
3,http://podtrac.com/pts/redirect.mp3/traffic.li...,4,10-part series Peak Card. I'm Mark Lindstamir...,Mark Lindstamire,48.28,67.14,Pretty Much Pop: A Culture Podcast,PMP#42: Star Trek Lives Long and Prospers (Int...,society,3390.0
4,http://podtrac.com/pts/redirect.mp3/traffic.li...,5,show .,NO_INFERRED_SPEAKER,67.14,67.56,Pretty Much Pop: A Culture Podcast,PMP#42: Star Trek Lives Long and Prospers (Int...,society,3390.0
...,...,...,...,...,...,...,...,...,...,...
2950,https://traffic.libsyn.com/secure/worldrider/W...,152,do one more toast. I'm,Ryan Pyle,2870.24,2871.25,WorldRider Journeys Around The World On A Moto...,#39 Ryan Pyle | Adventurers & Travelers Face N...,society,3017.0
2951,https://traffic.libsyn.com/secure/worldrider/W...,153,running out of whiskey,NO_INFERRED_SPEAKER,2871.25,2872.21,WorldRider Journeys Around The World On A Moto...,#39 Ryan Pyle | Adventurers & Travelers Face N...,society,3017.0
2952,https://traffic.libsyn.com/secure/worldrider/W...,154,". These flasks are small, Alan. These flasks a...",Ryan Pyle,2872.21,2876.08,WorldRider Journeys Around The World On A Moto...,#39 Ryan Pyle | Adventurers & Travelers Face N...,society,3017.0
2953,https://traffic.libsyn.com/secure/worldrider/W...,155,"you a bigger one, but I got to get that logo ...",Alan Collier,2876.08,3029.00,WorldRider Journeys Around The World On A Moto...,#39 Ryan Pyle | Adventurers & Travelers Face N...,society,3017.0
